In [1]:
import pandas as pd
import numpy as np

In [2]:
df1=pd.read_excel('online_retail_II.xlsx',sheet_name='Year 2009-2010')
df2=pd.read_excel('online_retail_II.xlsx',sheet_name='Year 2010-2011')

# Combine datasets
df=pd.concat([df1,df2],ignore_index=True)
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Dataset Overview

In [3]:
print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

df.info()

Rows : 1067371
Columns : 8
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [26]:
# MISSING VALUE ANALYSIS
missing_values=df.isnull().sum()
missing_values[missing_values>0].sort_values(ascending=False)

Customer ID    243007
Description      4382
dtype: int64

In [27]:
# DUPLICATE RECORD ANALYSIS
duplicate_count=df.duplicated().sum()
print("Duplicate Records:",duplicate_count)
duplicates=df[df.duplicated()]
duplicates.head()

Duplicate Records: 34335


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom,5.10
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom,7.80
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75


## Converted Required Columns To NumPy

In [28]:
# Convert Required Columns to NumPy Arrays

quantity = df["Quantity"].to_numpy()

price = df["Price"].to_numpy()

customer = df["Customer ID"].to_numpy()

country = df["Country"].to_numpy()

product = df["Description"].to_numpy()

invoice = df["Invoice"].astype(str).to_numpy()

# From this point onward, analysis is performed mainly using NumPy arrays.
# This improves performance and satisfies assignment requirements.

In [29]:
# REVENUE CALCULATION USING NUMPY
revenue = quantity * price
df['Revenue'] = revenue
df[['Quantity','Price','Revenue']].head()

,Quantity,Price,Revenue
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


In [30]:
# MISSING CUSTOMER IDs
missing_customer=df[df['Customer ID'].isnull()]
print("Missing Customer IDs:",len(missing_customer))
missing_customer[['Invoice','Customer ID','Description','Quantity','Price','Country']].head()

Missing Customer IDs: 243007


,Invoice,Customer ID,Description,Quantity,Price,Country
263,489464,NaN,85123a mixed,-96,0.00,United Kingdom
283,489463,NaN,short,-240,0.00,United Kingdom
284,489467,NaN,21733 mixed,-192,0.00,United Kingdom
470,489521,NaN,NaN,-50,0.00,United Kingdom
577,489525,NaN,BLUE PULL BACK RACING CAR,1,0.55,United Kingdom


## Missing Value Audit

In [31]:
print("Missing Customer IDs:",np.sum(pd.isna(customer)))
print("Missing Product Descriptions:",np.sum(pd.isna(product)))
print("Missing Countries:",np.sum(pd.isna(country)))

Missing Customer IDs: 243007
Missing Product Descriptions: 4382
Missing Countries: 0


## RETURNED TRANSACTIONS

In [36]:
# RETURN ANALYSIS

returns_mask = quantity < 0

print(
    "Returned Transactions:",
    np.sum(returns_mask)
)

returned_records = df[
    returns_mask
]

returned_records[
    [
        'Invoice',
        'Description',
        'Quantity',
        'Price'
    ]
].head()

Returned Transactions: 22950


,Invoice,Description,Quantity,Price
178,C489449,PAPER BUNTING WHITE LACE,-12,2.95
179,C489449,CREAM FELT EASTER EGG BASKET,-6,1.65
180,C489449,POTTING SHED SOW 'N' GROW SET,-4,4.25
181,C489449,POTTING SHED TWINE,-6,2.10
182,C489449,PAPER CHAIN KIT RETRO SPOT,-12,2.95


## CANCELLED INVOICES

In [33]:
cancelled=df[df['Invoice'].astype(str).str.startswith('C')]
print("Cancelled Invoices:",len(cancelled))
cancelled[['Invoice','Description','Quantity','Price']].head()

Cancelled Invoices: 19494


,Invoice,Description,Quantity,Price
178,C489449,PAPER BUNTING WHITE LACE,-12,2.95
179,C489449,CREAM FELT EASTER EGG BASKET,-6,1.65
180,C489449,POTTING SHED SOW 'N' GROW SET,-4,4.25
181,C489449,POTTING SHED TWINE,-6,2.10
182,C489449,PAPER CHAIN KIT RETRO SPOT,-12,2.95


## INVALID PRICE RECORDS

In [37]:
invalid_price = price <= 0

print(
    "Invalid Price Records:",
    np.sum(invalid_price)
)

df[
    invalid_price
][
    [
        'Invoice',
        'Description',
        'Price'
    ]
].head()

Invalid Price Records: 6207


,Invoice,Description,Price
263,489464,85123a mixed,0.0
283,489463,short,0.0
284,489467,21733 mixed,0.0
470,489521,NaN,0.0
3114,489655,NaN,0.0


## Unusual HIGH QUANTITY ORDERS

In [40]:
high_quantity = quantity > 1000

print(
    "High Quantity Orders:",
    np.sum(high_quantity)
)

df[
    high_quantity
][
    [
        'Invoice',
        'Description',
        'Quantity',
        'Price'
    ]
].head()

High Quantity Orders: 352


,Invoice,Description,Quantity,Price
7302,490018,PACK OF 12 WOODLAND TISSUES,4320,0.25
7303,490018,PACK OF 12 SKULL TISSUES,5184,0.25
7304,490018,PACK OF 12 PINK PAISLEY TISSUES,4008,0.25
7305,490018,PACK OF 12 RED SPOTTY TISSUES,4008,0.25
17384,490758,NaN,3000,0.00


In [14]:
# REVENUE COMPARISON
gross_revenue=df['Revenue'].sum()
valid_revenue=df[(df['Quantity']>0)&(df['Price']>0)&(~df['Invoice'].astype(str).str.startswith('C'))]['Revenue'].sum()
print("Gross Revenue:",round(gross_revenue,2))
print("Valid Revenue:",round(valid_revenue,2))
print("Revenue Difference:",round(gross_revenue-valid_revenue,2))

Gross Revenue: 19287250.57
Valid Revenue: 20972594.57
Revenue Difference: -1685344.0


## Top Revenue Products

In [15]:
product_summary = (
    df[(df['Quantity'] > 0) & (df['Price'] > 0)]
    .groupby('Description')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(product_summary)

Description
REGENCY CAKESTAND 3 TIER              344563.25
Manual                                341104.90
DOTCOM POSTAGE                        322657.48
WHITE HANGING HEART T-LIGHT HOLDER    266923.55
PAPER CRAFT , LITTLE BIRDIE           168469.60
JUMBO BAG RED RETROSPOT               150935.56
PARTY BUNTING                         149187.05
ASSORTED COLOUR BIRD ORNAMENT         132187.92
POSTAGE                               127597.42
PAPER CHAIN KIT 50'S CHRISTMAS        123141.54
Name: Revenue, dtype: float64

## Top Revenue Countries

In [16]:
country_summary = (
    df[(df['Quantity'] > 0) & (df['Price'] > 0)]
    .groupby('Country')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(country_summary)

Country
United Kingdom    1.787135e+07
EIRE              6.644318e+05
Netherlands       5.542323e+05
Germany           4.312625e+05
France            3.569446e+05
Australia         1.699681e+05
Spain             1.091785e+05
Switzerland       1.010113e+05
Sweden            9.190372e+04
Denmark           6.986219e+04
Name: Revenue, dtype: float64

## High Value Customers

In [17]:
customer_summary = (
    df[
        (df['Quantity'] > 0) &
        (df['Price'] > 0) &
        (df['Customer ID'].notna())
    ]
    .groupby('Customer ID')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(customer_summary)

Customer ID
18102.0    608821.65
14646.0    528602.52
14156.0    313946.37
14911.0    295972.63
17450.0    246973.09
13694.0    196482.81
17511.0    175603.55
16446.0    168472.50
16684.0    147142.77
12415.0    144458.37
Name: Revenue, dtype: float64

## Extreme Revenue Orders

In [41]:
threshold = np.quantile(
    revenue,
    0.99
)

extreme_orders = revenue > threshold

print(
    "Extreme Revenue Orders:",
    np.sum(extreme_orders)
)

df[
    extreme_orders
][
    [
        'Invoice',
        'Description',
        'Quantity',
        'Price',
        'Revenue'
    ]
].head()

Extreme Revenue Orders: 10603


,Invoice,Description,Quantity,Price,Revenue
67,489438,CHARLIE + LOLA BISCUITS TINS,60,6.38,382.8
68,489438,CHARLIE AND LOLA FIGURES TINS,60,6.40,384.0
95,489441,SCOTTIE DOG HOT WATER BOTTLE,48,4.25,204.0
176,489448,GOLD WINE GOBLET,48,4.25,204.0
282,489465,ASSORTED COLOUR BIRD ORNAMENT,160,1.45,232.0


## SALES AUDIT KPI SUMMARY

In [19]:
print("\n===== SALES AUDIT KPI SUMMARY =====\n")
print("Total Transactions:",len(df))
print("Duplicate Records:",duplicate_count)
print("Returned Transactions:", np.sum(returns_mask))
print("Cancelled Invoices:",len(cancelled))
print("Missing Customer IDs:",len(missing_customer))
print("Valid Revenue:",round(valid_revenue,2))


===== SALES AUDIT KPI SUMMARY =====

Total Transactions: 1067371
Duplicate Records: 34335
Returned Transactions: 22950
Cancelled Invoices: 19494
Missing Customer IDs: 243007
Valid Revenue: 20972594.57


## Business Report

1. Sales revenue is strong but includes risky transactions.
2. Returns and cancellations are present in the dataset.
3. Missing customer IDs affect customer analysis.
4. Some transactions contain unusual quantities or prices.
5. Revenue is concentrated among a small number of products.
6. Revenue is concentrated among a few countries and customers.
7. Extreme values can influence KPI calculations.